# Fine-Tuning LLaMA 3.2 for PCOS — Ovula FYP

This is the fine-tuning notebook for our FYP project Ovula. The idea here is to take a small, open-source LLaMA model and train it specifically on PCOS-related Q&A data so it gives better, more contextual answers than a generic model would.

We're using [Unsloth](https://github.com/unslothai/unsloth) because it makes fine-tuning significantly faster and uses less VRAM — which matters a lot on a free Colab T4.

**To run this:**
1. Open in Google Colab
2. Runtime > Change runtime type > T4 GPU (free tier is fine)
3. Upload `pcos_training_complete.jsonl` via the Files sidebar on the left
4. Run all cells top to bottom
5. Download the `.gguf` file that gets exported at the end
6. Drop it into the `ai-models/` folder and register it with Ollama locally

Training takes around 30–60 minutes depending on the dataset size and how busy Colab is.

## 1. Install Unsloth

Unsloth patches HuggingFace's internals to speed up training. The install is a bit verbose but this is the recommended way for Colab.

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("Unsloth installed")

## 2. Imports

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Config

These are the main knobs to tune. `max_steps=100` is good enough for a quick experiment — bump it up if you want the model to actually absorb more of the dataset. We kept batch size small because T4 only has 15GB VRAM.

In [ ]:
max_seq_length = 2048
dtype = None  # auto — Float16 for T4, BFloat16 for Ampere+
load_in_4bit = True  # 4-bit quantization to fit in VRAM

per_device_train_batch_size = 2
gradient_accumulation_steps = 4  # effective batch = 2 * 4 = 8
warmup_steps = 5
max_steps = 100  # increase for longer training
learning_rate = 2e-4

print(f"Seq length: {max_seq_length}, Steps: {max_steps}, LR: {learning_rate}")

## 4. Load Base Model

We went with Llama 3.2 1B Instruct as the base. It's small enough to fine-tune on Colab's free tier and already knows how to follow instructions, so we're just adding PCOS domain knowledge on top.

In [ ]:
print("Loading base model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Loaded: {model.config._name_or_path}")
print(f"Vocab size: {len(tokenizer)}")

## 5. Apply LoRA

Instead of updating all 1B parameters (which would be way too slow and memory-heavy), LoRA injects small trainable matrices into the attention layers. We're targeting the standard attention + MLP projection layers with rank 16, which is a decent middle ground.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

## 6. Upload & Load Dataset

Upload `pcos_training_complete.jsonl` using the file browser on the left sidebar in Colab (the folder icon), then run this cell.

In [ ]:
import os

dataset_file = "pcos_training_complete.jsonl"

if not os.path.exists(dataset_file):
    print("Dataset not found. Upload it first via the Files sidebar.")
    print("Or run: from google.colab import files; files.upload()")
else:
    dataset = load_dataset("json", data_files=dataset_file, split="train")
    print(f"Loaded {len(dataset)} training examples")
    print("\nSample:")
    print(dataset[0])

## 7. Format Dataset

The trainer expects a single text field per example. We're using the Alpaca prompt format since it works well for instruction fine-tuning. Our dataset has two possible schemas (prompt/completion or instruction/output), so we handle both.

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []

    if "prompt" in examples and "completion" in examples:
        for prompt, completion in zip(examples["prompt"], examples["completion"]):
            question = prompt.replace("User: ", "").replace("\nAssistant:", "").strip()
            texts.append(alpaca_prompt.format(question, "", completion) + EOS_TOKEN)

    elif "instruction" in examples and "output" in examples:
        inputs = examples.get("input", [""] * len(examples["instruction"]))
        for instruction, inp, output in zip(examples["instruction"], inputs, examples["output"]):
            texts.append(alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print("Formatted. Preview:")
print(dataset[0]["text"][:500] + "...")

## 8. Set Up Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=learning_rate,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Trainer ready.")

## 9. GPU Check

Just making sure we actually have a GPU before kicking off training. If this says no GPU, go to Runtime > Change runtime type and pick T4.

In [ ]:
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
    total = round(gpu.total_memory / 1024**3, 2)
    print(f"GPU: {gpu.name}")
    print(f"VRAM: {total} GB total, {reserved} GB reserved, {round(total - reserved, 2)} GB free")
else:
    print("No GPU! Go to Runtime > Change runtime type > T4 GPU")

## 10. Train

This is the actual fine-tuning step. Watch the loss — it should go down fairly steadily. If it plateaus early, consider increasing `max_steps` or adjusting the learning rate.

In [ ]:
print("Starting training...")

trainer_stats = trainer.train()

print(f"Done! Steps: {trainer_stats.global_step}, Loss: {trainer_stats.training_loss:.4f}")

## 11. Quick Test

Let's throw a basic PCOS question at the model and see what it says. This is just a sanity check — the response quality should noticeably improve compared to the base model.

In [ ]:
FastLanguageModel.for_inference(model)

test_question = "What is PCOS and how common is it?"
test_prompt = alpaca_prompt.format(test_question, "", "")

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = response.split("### Response:")[-1].strip()

print(f"Q: {test_question}\n")
print(f"A: {response}")

## 12. Export to GGUF

We need to convert the model to GGUF format so Ollama can load it. Q8_0 quantization gives a decent balance between file size and response quality — we found it worked well in testing.

In [ ]:
print("Converting to GGUF (Q8_0)... this takes a few minutes")

model.save_pretrained_gguf(
    "pcos_assistant_model",
    tokenizer,
    quantization_method="q8_0"
)

print("Done!")
!ls -lh pcos_assistant_model/

## 13. Download the Model File

This will trigger a browser download for the `.gguf` file. Move it to `ai-models/` in the project after downloading.

In [ ]:
from google.colab import files
import os

gguf_files = [f for f in os.listdir("pcos_assistant_model") if f.endswith(".gguf")]

if gguf_files:
    gguf_path = f"pcos_assistant_model/{gguf_files[0]}"
    size_gb = os.path.getsize(gguf_path) / (1024**3)
    print(f"Downloading {gguf_files[0]} ({size_gb:.2f} GB)")
    files.download(gguf_path)
else:
    print("No GGUF file found — did the export step finish?")

## 14. Using the Model Locally with Ollama

Once you've downloaded the `.gguf` file, drop it in `ai-models/` and follow these steps:

**Create the Modelfile** (there's already one in `ai-models/Modelfile` — just update the FROM path):

```
FROM ./pcos_assistant_model-unsloth.Q8_0.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9

SYSTEM "You are a helpful PCOS healthcare assistant. Give accurate, empathetic, evidence-based answers about PCOS. Always remind users to consult a doctor for personal medical decisions."
```

**Register with Ollama:**

```bash
cd ai-models
ollama create pcos-assistant -f Modelfile
```

**Test it:**

```bash
ollama run pcos-assistant "What is PCOS?"
```

**Hook it into the backend** — update `backend/.env`:

```
MODEL_TYPE=ollama_finetuned
OLLAMA_FINETUNED_MODEL=pcos-assistant
```

Then restart the backend and the chat screen in the app will use the fine-tuned model.